In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# 目标网页URL
url = "https://amr.sz.gov.cn/xxgk/qt/ztlm/sjfb/tjfx/cqsj/content/post_12621622.html"

# 请求头，模拟浏览器访问
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

try:
    # 发送HTTP请求获取网页内容
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()  # 检查请求是否成功
    response.encoding = 'utf-8'  # 设置编码，确保中文正确显示
    html = response.text

    # 使用BeautifulSoup解析HTML
    soup = BeautifulSoup(html, 'html.parser')

    # 找到表格（增加容错，确保能找到表格）
    table = soup.find('table')
    if not table:
        raise ValueError("未在页面中找到表格")

    # 提取所有行数据
    rows = table.find_all('tr')
    if len(rows) < 2:
        raise ValueError("表格数据过少，无法解析")

    # 优化：提取表头（取第一行的所有单元格，确保列数正确）
    headers = []
    header_cells = rows[0].find_all(['th', 'td'])  # 兼容th/td两种表头标签
    for cell in header_cells:
        header_text = re.sub(r'\s+', ' ', cell.text.strip())
        headers.append(header_text)

    # 提取表格数据（从第二行开始）
    data = []
    for row in rows[1:]:
        row_data = []
        cells = row.find_all('td')
        
        # 确保每行的列数和表头一致（补空值）
        for i in range(len(headers)):
            if i < len(cells):
                cell_text = re.sub(r'\s+', ' ', cells[i].text.strip())
                # 尝试转换为数字（兼容整数/浮点数）
                try:
                    # 先尝试转整数，失败则转浮点数
                    row_data.append(int(cell_text))
                except ValueError:
                    try:
                        row_data.append(float(cell_text))
                    except ValueError:
                        row_data.append(cell_text)
            else:
                row_data.append("")  # 列数不足时补空值

        data.append(row_data)

    # 创建DataFrame（确保列数匹配）
    df = pd.DataFrame(data, columns=headers)

    # 保存为CSV文件
    output_filename = "2025年深圳市各区专利授权情况.csv"
    df.to_csv(output_filename, index=False, encoding='utf-8-sig')  # 兼容Excel中文显示

    print(f"✅ 数据已成功保存为 {output_filename}")
    print(f"📊 数据预览：")
    print(df.head())  # 打印前5行预览

except requests.exceptions.RequestException as e:
    print(f"❌ 请求失败：{e}")
except ValueError as e:
    print(f"❌ 数据解析失败：{e}")
except Exception as e:
    print(f"❌ 程序运行出错：{e}")

✅ 数据已成功保存为 2025年深圳市各区专利授权情况.csv
📊 数据预览：
  2025年深圳市各区专利授权情况汇总表
0               2025年
1                  1月
2                  2月
3                  3月
4                  4月
